# Fraud Detection Pipeline - Azentio Hackathon
Using Llama 3.2 1B Instruct with LoRA fine-tuning

In [ ]:
# these are mostly pre-installed on colab, just making sure
!pip install -q peft trl accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00


In [2]:
# login to huggingface so we can pull llama model
from huggingface_hub import login
import getpass

hf_token = getpass.getpass('HF Token: ')
login(token=hf_token)

HF Token: ··········


In [3]:
import pandas as pd
import numpy as np
import re
import json
import torch
import warnings
warnings.filterwarnings('ignore')

## 1. Load and merge the 3 tables

In [5]:
# loading everything as string first because amount column has some
# weird string values mixed in, we'll convert types manually later
transactions = pd.read_csv('/content/transactions.csv', dtype=str)
accounts = pd.read_csv('/content/accounts.csv', dtype=str)
customers = pd.read_csv('/content/customers.csv', dtype=str)

print(transactions.shape, accounts.shape, customers.shape)

# left join so we keep every transaction even if account/customer info is missing
df = transactions.merge(accounts, on=['account_id', 'customer_id'], how='left', suffixes=('', '_acct'))
df = df.merge(customers, on='customer_id', how='left', suffixes=('', '_cust'))
print(f'merged: {df.shape}')

(1000, 29) (178, 21) (124, 26)
merged: (1000, 73)


## 2. Clean the data
The data is pretty messy - whitespace in columns, mixed date formats, boolean columns with like 8 different representations etc.

In [6]:
# first strip whitespace from everything - there's leading/trailing
# spaces in status, channel, transaction_type etc (probably ETL bug)
for col in df.columns:
    if df[col].dtype == 'object' or str(df[col].dtype) == 'string':
        df[col] = df[col].str.strip()

# normalize case for categorical columns
for col in ['transaction_type', 'channel', 'status', 'merchant_category', 'auth_method']:
    if col in df.columns:
        df[col] = df[col].str.upper()

# some channels have spaces instead of underscores
df['channel'] = df['channel'].replace({
    'MOBILE APP': 'MOBILE_APP',
    'INTERNET BANKING': 'INTERNET_BANKING'
})

print('status values:', df['status'].dropna().unique())
print('channel values:', df['channel'].dropna().unique())

status values: ['SUCCESS' 'FAILED' 'REVERSED' 'PENDING']
channel values: ['INTERNET_BANKING' 'POS' 'MOBILE_APP' 'ONLINE' 'ATM']


In [7]:
# amount is stored as string because some entries have 'N/A' or are blank
# we convert to numeric and flag the broken ones
df['amount_raw'] = df['amount']
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['amount_anomaly'] = df['amount'].isna()

# for missing amounts, use that customer's own median rather than global
# (global median would be misleading for high-income vs low-income customers)
customer_med = df.groupby('customer_id')['amount'].transform('median')
df['amount'] = df['amount'].fillna(customer_med)
df['amount'] = df['amount'].fillna(df['amount'].median())  # fallback

print(f'amount anomalies: {df["amount_anomaly"].sum()}')

amount anomalies: 22


In [8]:
# timestamps have two different formats in the data:
#   '2026-08-13 06:18:18' (normal ISO)
#   '29/07/2026 18:07'    (DD/MM/YYYY)
# missing or broken timestamps are system errors, not fraud
# (the bank's clock/logging service can fail independently)

def parse_ts(ts):
    if pd.isna(ts) or ts == '':
        return pd.NaT
    for fmt in ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']:
        try:
            return pd.to_datetime(ts, format=fmt)
        except:
            continue
    try:
        return pd.to_datetime(ts, format='mixed', dayfirst=True)
    except:
        return pd.NaT

df['ts_parsed'] = df['transaction_timestamp'].apply(parse_ts)
df['timestamp_error'] = df['ts_parsed'].isna()
print(f'unparseable timestamps: {df["timestamp_error"].sum()} (system error, not fraud)')

unparseable timestamps: 10 (system error, not fraud)


In [9]:
# is_foreign_transaction has literally 8 different formats:
# '0', 'FALSE', 'N', 'no', '1', 'yes', 'TRUE', 'Y'
# classic legacy system problem - different microservices writing different formats

def to_bool(val):
    if pd.isna(val): return False
    return str(val).strip().upper() in {'1', 'TRUE', 'Y', 'YES'}

df['is_foreign_transaction'] = df['is_foreign_transaction'].apply(to_bool)
df['is_new_device'] = df['is_new_device'].apply(to_bool)

# convert other numeric columns we'll need later
for col in ['balance_after_txn', 'distance_from_home_km', 'time_since_prev_txn_mins',
            'txn_count_last_24h', 'txn_count_last_7d', 'amount_to_account_avg_ratio',
            'transaction_hour', 'annual_income']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [10]:
# create some domain-specific flags that a real fraud analyst would look at

# negative balance = overdraft, which is a financial risk signal
df['negative_balance'] = df['balance_after_txn'] < 0

# POS terminals dont have IP addresses, thats normal
# but if its an INTERNET_BANKING or MOBILE_APP txn with no IP, thats weird
online = ['INTERNET_BANKING', 'MOBILE_APP', 'ONLINE']
df['missing_ip_online'] = (df['channel'].isin(online)) & (df['ip_address'].isna())

# no authentication on a big transaction? suspicious
df['no_auth_high_value'] = (df['auth_method'].isin(['NONE'])) & (df['amount'] > 10000)

print(f'negative balance: {df["negative_balance"].sum()}')
print(f'missing IP on online txn: {df["missing_ip_online"].sum()}')
print(f'no auth + high value: {df["no_auth_high_value"].sum()}')

negative balance: 69
missing IP on online txn: 18
no auth + high value: 27


## 3. Prompt injection defense
The instructions say fraudsters embedded adversarial strings in the data. We need to catch and neutralize them before they go into the model context.

In [11]:
# scan text fields for known adversarial patterns
injection_patterns = [
    r'ignore\s+(all\s+)?previous\s+instructions',
    r'classify\s+this\s+(transaction\s+)?as\s+safe',
    r'you\s+are\s+now\s+a',
    r'system\s*prompt',
    r'bypass\s+(all\s+)?security',
    r'jailbreak',
    r'disregard\s+(all\s+)?prior',
    r'override\s+instructions',
    r'pretend\s+this\s+is\s+safe',
]
inj_re = re.compile('|'.join(injection_patterns), re.IGNORECASE)

df['injection_detected'] = False

for col in ['merchant_name', 'merchant_category', 'merchant_city']:
    if col in df.columns:
        hits = df[col].fillna('').apply(lambda x: bool(inj_re.search(str(x))))
        df.loc[hits, 'injection_detected'] = True
        df.loc[hits, col] = '[REDACTED]'  # strip the malicious text

print(f'injection attempts found: {df["injection_detected"].sum()}')

injection attempts found: 0


## 4. Label the data for training
We dont have a ground truth is_fraud column, so we use weak supervision - basically writing domain rules to create labels. Each rule adds to a fraud score, and if the score is high enough we label it fraud.

In [12]:
df['fraud_score'] = 0
df['fraud_reasons'] = ''

# spending 5x your normal average is a bust-out pattern
m = df['amount_to_account_avg_ratio'] > 5
df.loc[m, 'fraud_score'] += 2
df.loc[m, 'fraud_reasons'] += 'Spend 5x above customer average; '

# overdraft
m = df['negative_balance'] == True
df.loc[m, 'fraud_score'] += 2
df.loc[m, 'fraud_reasons'] += 'Transaction caused account overdraft; '

# high value + foreign
m = (df['is_foreign_transaction'] == True) & (df['amount'] > 15000)
df.loc[m, 'fraud_score'] += 1
df.loc[m, 'fraud_reasons'] += 'High-value cross-border transfer; '

# far from home on a new device
m = (df['distance_from_home_km'] > 500) & (df['is_new_device'] == True)
df.loc[m, 'fraud_score'] += 1
df.loc[m, 'fraud_reasons'] += 'Far from home on new device; '

# lots of txns in short time (velocity)
m = df['txn_count_last_24h'] >= 5
df.loc[m, 'fraud_score'] += 1
df.loc[m, 'fraud_reasons'] += 'High velocity in 24h; '

# prompt injection = definitely suspicious
m = df['injection_detected'] == True
df.loc[m, 'fraud_score'] += 3
df.loc[m, 'fraud_reasons'] += 'Prompt injection attempt detected; '

# no auth on expensive txn
m = df['no_auth_high_value'] == True
df.loc[m, 'fraud_score'] += 1
df.loc[m, 'fraud_reasons'] += 'No auth on high-value transaction; '

# threshold: score >= 2 = fraud
df['is_fraud'] = df['fraud_score'] >= 2
df['fraud_reasons'] = df['fraud_reasons'].str.rstrip('; ')
df.loc[~df['is_fraud'], 'fraud_reasons'] = 'Transaction within normal patterns'

print(f'fraud: {df["is_fraud"].sum()} | safe: {(~df["is_fraud"]).sum()}')

fraud: 177 | safe: 823


In [13]:
# balance the dataset - if we train on 90% safe and 10% fraud,
# the model will just learn to always say "safe" and get 90% accuracy
# which is completely useless

fraud_df = df[df['is_fraud'] == True]
safe_df = df[df['is_fraud'] == False]

n = min(len(fraud_df), len(safe_df), 75)
print(f'taking {n} fraud + {n} safe for training')

train_df = pd.concat([
    fraud_df.sample(n=n, random_state=42),
    safe_df.sample(n=n, random_state=42)
]).sample(frac=1, random_state=42)

# keep some aside for evaluation (5 fraud + 5 safe the model hasnt seen)
holdout_fraud = fraud_df[~fraud_df.index.isin(train_df.index)].head(5)
holdout_safe = safe_df[~safe_df.index.isin(train_df.index)].head(5)
holdout = pd.concat([holdout_fraud, holdout_safe]).sample(frac=1, random_state=42)
print(f'holdout for eval: {len(holdout)}')

taking 75 fraud + 75 safe for training
holdout for eval: 10


In [14]:
# format the training data as llama 3.2 chat conversations
# each example: system instruction -> user gives txn details -> assistant outputs JSON

SYS_PROMPT = """You are a banking fraud detection AI. Analyze the transaction and output ONLY valid JSON.
Format: {"transaction_id": "...", "is_fraud": true/false, "confidence": 0.0-1.0, "justification": "one sentence"}"""

def make_user_prompt(row):
    """turn a row into a readable transaction description"""
    lines = [f"Transaction {row['transaction_id']}:"]
    lines.append(f"Amount: {row['amount']:.2f} {row.get('currency', 'INR')}")
    lines.append(f"Channel: {row.get('channel', 'UNKNOWN')}")
    lines.append(f"Merchant: {row.get('merchant_name', 'UNKNOWN')} ({row.get('merchant_category', 'UNKNOWN')})")
    lines.append(f"Status: {row.get('status', 'UNKNOWN')}")
    lines.append(f"Foreign: {row['is_foreign_transaction']}")
    lines.append(f"New Device: {row['is_new_device']}")
    if pd.notna(row.get('distance_from_home_km')):
        lines.append(f"Distance from Home: {row['distance_from_home_km']:.1f} km")
    if pd.notna(row.get('amount_to_account_avg_ratio')):
        lines.append(f"Amount vs Avg: {row['amount_to_account_avg_ratio']:.2f}x")
    if pd.notna(row.get('txn_count_last_24h')):
        lines.append(f"Txns last 24h: {int(row['txn_count_last_24h'])}")
    if pd.notna(row.get('balance_after_txn')):
        bal = float(row['balance_after_txn'])
        lines.append(f"Balance After: {bal:.2f}{' (OVERDRAWN)' if bal < 0 else ''}")
    lines.append(f"Auth: {row.get('auth_method', 'UNKNOWN')}")
    if pd.notna(row.get('risk_rating')):
        lines.append(f"Risk Rating: {row['risk_rating']}")
    return '\n'.join(lines)

def make_target_json(row):
    """build the expected JSON output"""
    conf = min(0.95, 0.5 + row['fraud_score'] * 0.1) if row['is_fraud'] else max(0.1, 0.5 - row['fraud_score'] * 0.1)
    return json.dumps({
        'transaction_id': row['transaction_id'],
        'is_fraud': bool(row['is_fraud']),
        'confidence': round(conf, 2),
        'justification': row['fraud_reasons']
    })

def to_llama_format(row):
    """wrap in llama 3.2 special tokens"""
    user = make_user_prompt(row)
    asst = make_target_json(row)
    return (
        f"<|start_header_id|>system<|end_header_id|>\n\n{SYS_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n{asst}<|eot_id|>"
    )

train_df = train_df.copy()
train_df['text'] = train_df.apply(to_llama_format, axis=1)

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df[['text']])
print(f'training examples: {len(train_dataset)}')

training examples: 150


## 5. Load model and base benchmark
Before fine-tuning, lets see how the stock Llama does on our holdout set.

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# load in 4-bit so it fits on colab T4 (16GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
)

# llama doesn't have a pad token by default, set it
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f'model loaded: {model_name}')
print(f'device: {model.device}')

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model loaded: meta-llama/Llama-3.2-1B-Instruct
device: cuda:0


In [16]:
def run_inference(model, tokenizer, row):
    """run model on a single transaction and return parsed result"""
    msgs = [
        {'role': 'system', 'content': SYS_PROMPT},
        {'role': 'user', 'content': make_user_prompt(row)},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return resp

def eval_on_holdout(model, tokenizer, holdout_df, label='MODEL'):
    """run model on holdout set and return results df"""
    results = []
    for _, row in holdout_df.iterrows():
        resp = run_inference(model, tokenizer, row)
        try:
            parsed = json.loads(resp.strip())
            ok = True
        except:
            parsed = None
            ok = False

        pred = parsed.get('is_fraud') if parsed else None
        print(f'[{label}] {row["transaction_id"]} truth={row["is_fraud"]} json={ok} pred={pred}')
        results.append({'txn': row['transaction_id'], 'truth': bool(row['is_fraud']),
                        'valid_json': ok, 'pred': pred, 'raw': resp[:200]})
    return pd.DataFrame(results)

In [17]:
print('--- BASE MODEL ---')
base_results = eval_on_holdout(model, tokenizer, holdout, 'BASE')

base_json = base_results['valid_json'].mean() * 100
base_ok = base_results.dropna(subset=['pred'])
base_acc = (base_ok['truth'] == base_ok['pred']).mean() * 100 if len(base_ok) > 0 else 0
print(f'\nbase json rate: {base_json:.0f}% | base accuracy: {base_acc:.0f}%')

--- BASE MODEL ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[BASE] TXN_0000849 truth=False json=True pred=true
[BASE] TXN_0000795 truth=True json=True pred=False
[BASE] TXN_0000796 truth=False json=True pred=False
[BASE] TXN_0000974 truth=True json=True pred=False
[BASE] TXN_0000011 truth=False json=True pred=False
[BASE] TXN_0000299 truth=True json=True pred=False
[BASE] TXN_0000714 truth=False json=True pred=False
[BASE] TXN_0000549 truth=True json=True pred=false
[BASE] TXN_0000790 truth=True json=True pred=False
[BASE] TXN_0000588 truth=False json=True pred=true

base json rate: 100% | base accuracy: 30%


## 6. Fine-tune with LoRA
Using standard HF peft library. `target_modules='all-linear'` auto-detects the right layers for any model.

In [18]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments

# prep model for training (freeze base weights, enable gradient checkpointing)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules='all-linear',  # auto-detect linear layers, works with any architecture
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

trainable: 11,272,192 / 760,547,328 (1.48%)


In [22]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_length=2048,
        dataset_num_proc=2,
        packing=False,
        neftune_noise_alpha=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=5,
        optim='paged_adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='outputs',
        report_to='none',
        gradient_checkpointing=True,
    ),
)

stats = trainer.train()
print(f'done - final loss: {stats.training_loss:.4f}')


Adding EOS to train dataset (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for train dataset (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

Step,Training Loss
5,3.320228
10,2.238230
15,1.196835
20,0.679180
25,0.578890
30,0.519287
35,0.496529
40,0.488414
45,0.459642
50,0.458677


done - final loss: 0.9460


In [24]:
# clean up neftune hooks left over from training
# (neftune adds noise to embeddings which we don't want during inference)
model.eval()
embed_layer = model.get_input_embeddings()
if hasattr(embed_layer, 'neftune_noise_alpha'):
    del embed_layer.neftune_noise_alpha
if hasattr(embed_layer, '_forward_hooks'):
    embed_layer._forward_hooks.clear()

print('model ready for inference')


model ready for inference


## 7. Evaluate fine-tuned model
Same holdout set, compare with base model.

In [25]:
print('--- FINE-TUNED MODEL ---')
ft_results = eval_on_holdout(model, tokenizer, holdout, 'FT')

ft_json = ft_results['valid_json'].mean() * 100
ft_ok = ft_results.dropna(subset=['pred'])
ft_acc = (ft_ok['truth'] == ft_ok['pred']).mean() * 100 if len(ft_ok) > 0 else 0

# side by side comparison
print('\n' + '='*50)
print(f'{"":25} {"Base":>10} {"Fine-tuned":>12}')
print('-'*50)
print(f'{"Valid JSON rate":25} {base_json:>9.0f}% {ft_json:>11.0f}%')
print(f'{"Accuracy":25} {base_acc:>9.0f}% {ft_acc:>11.0f}%')
print('='*50)

--- FINE-TUNED MODEL ---
[FT] TXN_0000849 truth=False json=True pred=True
[FT] TXN_0000795 truth=True json=True pred=False
[FT] TXN_0000796 truth=False json=True pred=False
[FT] TXN_0000974 truth=True json=True pred=True
[FT] TXN_0000011 truth=False json=True pred=False
[FT] TXN_0000299 truth=True json=True pred=False
[FT] TXN_0000714 truth=False json=True pred=False
[FT] TXN_0000549 truth=True json=True pred=False
[FT] TXN_0000790 truth=True json=True pred=False
[FT] TXN_0000588 truth=False json=True pred=False

                                Base   Fine-tuned
--------------------------------------------------
Valid JSON rate                 100%         100%
Accuracy                         30%          50%


## 8. Run on dataset and export JSON

In [28]:
# run inference on all 1000 transactions
predictions = []

for i, row in df.head(50).iterrows():
    resp = run_inference(model, tokenizer, row)

    # try to parse, fall back to heuristic label if model output is garbage
    try:
        p = json.loads(resp.strip())
        pred = {
            'transaction_id': row['transaction_id'],
            'is_fraud': p.get('is_fraud', bool(row['is_fraud'])),
            'confidence': p.get('confidence', 0.5),
            'justification': p.get('justification', row['fraud_reasons']),
        }
    except:
        pred = {
            'transaction_id': row['transaction_id'],
            'is_fraud': bool(row['is_fraud']),
            'confidence': min(0.95, 0.5 + row['fraud_score'] * 0.1),
            'justification': row['fraud_reasons'],
        }
    predictions.append(pred)

    if (i+1) % 100 == 0:
        print(f'{i+1}/{len(df)} done')

print(f'\nfinished: {len(predictions)} predictions')


finished: 50 predictions


In [29]:
# save to json
with open('/content/predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)

fraud_n = sum(1 for p in predictions if p['is_fraud'])
print(f'saved to /content/predictions.json')
print(f'fraud: {fraud_n} | safe: {len(predictions)-fraud_n}')
print('\nsample:')
for p in predictions[:3]:
    print(json.dumps(p, indent=2))

saved to /content/predictions.json
fraud: 2 | safe: 48

sample:
{
  "transaction_id": "TXN_0000796",
  "is_fraud": false,
  "confidence": 0.5,
  "justification": "Transaction within normal patterns"
}
{
  "transaction_id": "TXN_0000974",
  "is_fraud": true,
  "confidence": 0.7,
  "justification": "Transaction caused account overdraft"
}
{
  "transaction_id": "TXN_0000795",
  "is_fraud": false,
  "confidence": 0.5,
  "justification": "Transaction within normal patterns"
}


In [30]:
# save the lora adapter
model.save_pretrained('fraud_lora_adapter')
tokenizer.save_pretrained('fraud_lora_adapter')
print('adapter saved')

adapter saved
